# Greedy-CD on ER Graphs

Standalone benchmark notebook for `dag_greedy_A` on random ER graphs.
It keeps the same evaluation protocol as the main ER benchmark notebook,
but only runs Greedy-CD.

In [1]:
# 1) Environment and imports
import os
import sys
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

print('Python   :', sys.version.split()[0])
print('Repo root:', repo_root)

from MEC import is_in_markov_equiv_class, get_skeleton, find_v_structures
from synthetic_dataset import SyntheticDataset
from coordinate_descent.cd_greedy_A import dag_greedy_A as greedy_cd_noepoch_fit

try:
    _tb = os.path.join(repo_root, 'toolbox')
    if _tb not in sys.path:
        sys.path.append(_tb)
    from cdt.metrics import SHD_CPDAG as _SHD_CPDAG
    HAS_CPDAG_SHD = True
except Exception as _e:
    _SHD_CPDAG = None
    HAS_CPDAG_SHD = False
    print('CPDAG-SHD backend unavailable (fallback):', _e)

print('Greedy-CD : OK')

Python   : 3.10.11
Repo root: c:\Users\super\DAG
c:\Users\super\DAG\experiments\notebooks\test
CPDAG-SHD backend unavailable (fallback): No module named 'GPUtil'
Greedy-CD : OK


In [ ]:
# 2) Helpers
def weight_to_binary_adj(W: np.ndarray, threshold: float = 0.05) -> np.ndarray:
    G = (np.abs(W) > threshold).astype(int)
    np.fill_diagonal(G, 0)
    return G


def cpdag_shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    if HAS_CPDAG_SHD:
        try:
            return float(_SHD_CPDAG(G_true.astype(int), G_est.astype(int)))
        except Exception:
            pass
    skel_true = get_skeleton(G_true)
    skel_est = get_skeleton(G_est)
    skel_diff = int(np.sum(np.abs(skel_true - skel_est)) // 2)
    v_diff = len(find_v_structures(G_true).symmetric_difference(find_v_structures(G_est)))
    return float(skel_diff + v_diff)


def shd_score(G_true: np.ndarray, G_est: np.ndarray) -> float:
    G_true = np.asarray(G_true, dtype=int)
    G_est = np.asarray(G_est, dtype=int)
    d, dist = G_true.shape[0], 0
    for i in range(d):
        for j in range(i + 1, d):
            if G_true[i, j] != G_est[i, j] or G_true[j, i] != G_est[j, i]:
                dist += 1
    return float(dist)


def make_row(d, n, trial_id, seed, alg, runtime_sec, G_true, G_est):
    mec = int(is_in_markov_equiv_class(G_true, G_est))
    shd = shd_score(G_true, G_est)
    cshd = cpdag_shd_score(G_true, G_est)
    return {
        'd': d,
        'n_samples': n,
        'trial_id': trial_id,
        'seed': seed,
        'algorithm': alg,
        'mec_match': mec,
        'shd': shd,
        'cpdag_shd': cshd,
        'n_edges_true': int(G_true.sum()),
        'n_edges_est': int(G_est.sum()),
        'runtime_sec': float(runtime_sec),
    }


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    if len(df) == 0:
        return pd.DataFrame()
    return (
        df.groupby(['algorithm', 'd', 'n_samples'], as_index=False)
        .agg(
            mec_match_mean=('mec_match', 'mean'),
            shd_mean=('shd', 'mean'),
            cpdag_shd_mean=('cpdag_shd', 'mean'),
            runtime_sec_mean=('runtime_sec', 'mean'),
            trials=('trial_id', 'count'),
        )
    )


def iter_trials(cfg: dict):
    rng = np.random.default_rng(cfg['seed'])
    for d in cfg['d_list']:
        for n in cfg['n_list']:
            seeds = rng.integers(0, 10**9, size=cfg['trials'])
            for trial_idx, seed in enumerate(seeds, start=1):
                dataset = SyntheticDataset(
                    n=n,
                    d=d,
                    graph_type='ER',
                    degree=cfg['degree'],
                    noise_type=cfg['noise_type'],
                    B_scale=cfg['b_scale'],
                    seed=int(seed),
                )
                X = dataset.X
                S = X.T @ X / X.shape[0]
                G_true = weight_to_binary_adj(dataset.B, threshold=0.0)
                yield d, n, trial_idx, int(seed), X, S, G_true


print('Helpers ready.')

Helpers ready.


: 

In [ ]:
# 3) Config
CFG = {
    'trials': 1,
    'seed': 42,
    'd_list': [20, 30, 40, 50],
    'n_list': [20000],
    'degree': 3.0,
    'noise_type': 'gaussian_nv',
    'b_scale': 1.0,
    'threshold': 0.05,
    'T_noepoch': 100000,
    'lambda_l0_list': [0.0, 0.1, 0.2],
    'out_dir': os.path.join(repo_root, 'experiments', 'results'),
    'tag': 'greedy_cd_er_20260331',
}
os.makedirs(CFG['out_dir'], exist_ok=True)

print('Config ready.')
print(f"  d_list   : {CFG['d_list']}")
print(f"  n_list   : {CFG['n_list']}")
print(f"  trials   : {CFG['trials']}")
print(f"  lambdas  : {CFG['lambda_l0_list']}")
print(f"  steps T  : {CFG['T_noepoch']}")

In [ ]:
# 4) Run Greedy-CD
rows = []
skip_logs = []

for d, n, trial_id, seed, X, S, G_true in iter_trials(CFG):
    for lam in CFG['lambda_l0_list']:
        alg = f'greedy_cd_noepoch_l0_{lam:.1f}'
        try:
            t0 = time.perf_counter()
            _, G_est, _ = greedy_cd_noepoch_fit(
                S=S,
                T=CFG['T_noepoch'],
                seed=seed,
                threshold=CFG['threshold'],
                lambda_l0=lam,
            )
            rt = time.perf_counter() - t0
            row = make_row(d, n, trial_id, seed, alg, rt, G_true, G_est)
        except Exception as e:
            skip_logs.append({
                'algorithm': alg,
                'd': d,
                'n_samples': n,
                'trial_id': trial_id,
                'reason': str(e),
            })
            print(f'[SKIP] {alg} d={d} n={n} trial={trial_id}: {e}')
            continue

        rows.append(row)
        print(
            f'[{alg}] d={d} n={n} trial={trial_id}  '
            f'mec={row["mec_match"]}  shd={row["shd"]:.0f}  '
            f'cpdag_shd={row["cpdag_shd"]:.0f}  rt={rt:.3f}s'
        )

df_trials_greedy_cd = pd.DataFrame(rows)
df_summary_greedy_cd = summarize(df_trials_greedy_cd)

print('\nGreedy-CD summary:')
display(df_summary_greedy_cd)

In [ ]:
# 5) Save results
if len(df_trials_greedy_cd) == 0:
    print('WARNING: no Greedy-CD results collected.')
else:
    ts_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    trials_path = os.path.join(CFG['out_dir'], f"{CFG['tag']}_trials_{ts_str}.csv")
    summary_path = os.path.join(CFG['out_dir'], f"{CFG['tag']}_summary_{ts_str}.csv")
    latest_trials_path = os.path.join(CFG['out_dir'], f"{CFG['tag']}_trials.csv")
    latest_summary_path = os.path.join(CFG['out_dir'], f"{CFG['tag']}_summary.csv")

    df_trials_greedy_cd.to_csv(trials_path, index=False)
    df_summary_greedy_cd.to_csv(summary_path, index=False)
    df_trials_greedy_cd.to_csv(latest_trials_path, index=False)
    df_summary_greedy_cd.to_csv(latest_summary_path, index=False)

    print(f'Trials  saved to: {trials_path}')
    print(f'Summary saved to: {summary_path}')
    print(f'Rows: {len(df_trials_greedy_cd)}  |  Skips: {len(skip_logs)}')

In [ ]:
# 6) Overall comparison across lambdas
if len(df_trials_greedy_cd) == 0:
    print('Run the Greedy-CD cell first.')
else:
    df_overall = (
        df_trials_greedy_cd
        .groupby('algorithm', as_index=False)
        .agg(
            mec_match_mean=('mec_match', 'mean'),
            shd_mean=('shd', 'mean'),
            cpdag_shd_mean=('cpdag_shd', 'mean'),
            runtime_sec_mean=('runtime_sec', 'mean'),
            trials=('trial_id', 'count'),
        )
        .sort_values('algorithm')
        .reset_index(drop=True)
    )
    display(df_overall)

In [ ]:
# 7) Plots
if len(df_trials_greedy_cd) == 0:
    print('Run the Greedy-CD cell first.')
else:
    n_vals = sorted(df_trials_greedy_cd['n_samples'].unique())
    algs = sorted(df_trials_greedy_cd['algorithm'].unique())

    for metric, ylabel, title in [
        ('mec_match', 'MEC match rate', 'Greedy-CD MEC Match Rate'),
        ('shd', 'SHD', 'Greedy-CD SHD'),
        ('runtime_sec', 'Runtime (s)', 'Greedy-CD Runtime'),
    ]:
        fig, axes = plt.subplots(1, len(n_vals), figsize=(6 * len(n_vals), 4), sharey=True, squeeze=False)
        for ci, n in enumerate(n_vals):
            ax = axes[0][ci]
            sub = df_trials_greedy_cd[df_trials_greedy_cd['n_samples'] == n]
            agg = sub.groupby(['d', 'algorithm'])[metric].mean().reset_index()
            for alg in algs:
                vals = agg[agg['algorithm'] == alg]
                ax.plot(vals['d'], vals[metric], marker='o', label=alg)
            ax.set_title(f'n={n}')
            ax.set_xlabel('d (nodes)')
            if ci == 0:
                ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=8, loc='best')
        fig.suptitle(title, fontsize=13)
        plt.tight_layout()
        plt.show()

In [ ]:
# 8) Skip log
if skip_logs:
    display(pd.DataFrame(skip_logs))
else:
    print('No skips.')